# Griffschrift: Kopf-Erkenner nachlernen (YOLO)

Dieses Notizbuch erzeugt mit dem Griffschrift-Wandler Trainingsbilder mit bekannter Lösung (runde Köpfe voll/hohl, Kreuz, Kreuz im Kreis, Dreieck voll/hohl, vorgesetztes Doppelkreuz nach Rosenzopf), verschlechtert sie wie Scans, Kopien und Handyfotos und lernt damit einen Erkenner. Am Ende wird `griffschrift_koepfe.zip` heruntergeladen (Modell als ONNX und PyTorch, Lernkurven).

**Vorher:** oben im Menü *Laufzeit → Laufzeittyp ändern → T4-GPU* wählen. Dann *Laufzeit → Alle ausführen*. Das Ganze dauert etwa zwei Stunden; das Fenster muss dabei offen bleiben.

Die Notenblätter der Testpersonen werden hier nicht verwendet, nur vom Wandler gesetzte Stücke (Sätze von Franz Fuchs und eigene Beispiele aus dem Repository).

In [ ]:
%%writefile /content/yolo_erzeuge_wandlerbilder.js
// Trainingsbilder für den Kopf-Erkenner (YOLO): der Wandler setzt Stücke als Griffschrift, je System ein PNG und eine Liste der Köpfe
// mit Rahmen und Form. Die Formen werden zufällig abgewandelt, damit das Modell alle Schreibweisen kennt:
//   0 runder Kopf voll, 1 runder Kopf hohl, 2 Kreuz, 3 Kreuz im Kreis, 4 Dreieck voll, 5 Dreieck hohl,
//   6 vorgesetztes Doppelkreuz (Rosenzopf; der Kopf rechts davon ist dann rund und zählt als Kreuzkopf)
// Für das Modell zählt nur die Form, nicht Reihe oder Ton; Kreuze werden darum zufällig zu Dreiecken oder zu runden Köpfen mit
// vorgesetztem Doppelkreuz umgezeichnet. Seitenaufbau und Fotofehler macht yolo_daten.py.
// Aufruf (im Ordner griffschrift-wandler einen Webserver starten: python3 -m http.server 8798):
//   node yolo_erzeuge_wandlerbilder.js AUSGABEORDNER STARTWERT ANZAHL_STUECKE
const {chromium}=require('playwright');const fs=require('fs');
const [out,seed0,N]=process.argv.slice(2);fs.mkdirSync(out,{recursive:true});
const URL=process.env.WANDLER||'http://127.0.0.1:8798/index.html?lang=de';
(async()=>{const b=await chromium.launch();let k=0;let seed=+seed0;
 const rnd=()=>{seed=(seed*9301+49297)%233280;return seed/233280;};
 const pg0=await (await b.newContext()).newPage();await pg0.goto(URL);await pg0.waitForTimeout(2500);
 const VM=await pg0.evaluate(()=>[...document.querySelectorAll('#demo option')].map(o=>o.value).filter(v=>v.startsWith('vm:')||v.startsWith('mid:')));
 await pg0.close();
 for(const dsf of [1.5,2,2.5]){
 const pg=await (await b.newContext({locale:'de-DE',viewport:{width:1100+Math.round(rnd()*300),height:1400},deviceScaleFactor:dsf,colorScheme:'light'})).newPage();
 pg.on('dialog',d=>d.dismiss());
 await pg.goto(URL);await pg.waitForTimeout(2500);
 for(let ii=0;ii<+N;ii++){const pc=VM[Math.floor(rnd()*VM.length)];
  const vo=['1','2','3','4'][Math.floor(rnd()*4)];
  try{
   await pg.evaluate(async([pc,vo])=>{const set=(id,v)=>{const s=document.getElementById(id);s.value=v;s.dispatchEvent(new Event('change',{bubbles:true}));};
     set('voices',vo);set('view','griff');const [kind,key]=pc.split(':');if(kind==='vm')await loadVM(key);else{const ds=document.getElementById('demo');ds.value=pc;await loadDemo(pc);}},[pc,vo]);
  }catch(e){continue;}
  await pg.waitForTimeout(1500);
  const n=await pg.evaluate(()=>document.querySelectorAll('#griff svg.gsys').length);
  // je Stück eine Schreibweise: 0 Fuchs (Kreuze), 1 Margetich (4. Reihe Dreieck), 2 Rosenzopf (vorgesetzte Doppelkreuze), 3 gemischt
  const art=Math.floor(rnd()*4);
  for(let i=0;i<n;i++){
   const gt=await pg.evaluate(([i,sd,art])=>{let s0=sd;const rnd=()=>{s0=(s0*9301+49297)%233280;return s0/233280;};
    const NS='http://www.w3.org/2000/svg';
    const s=document.querySelectorAll('#griff svg.gsys')[i];
    const sl=[...s.querySelectorAll('line.sl')].filter(l=>l.getAttribute('y1')===l.getAttribute('y2')).map(l=>+l.getAttribute('y1'));
    const ys=[...new Set(sl)].sort((a,b)=>a-b);if(ys.length<5)return null;const d=(ys[4]-ys[0])/4;
    const gray=Math.round(20+150*rnd());s.querySelectorAll('line.sl').forEach(l=>{l.style.stroke=`rgb(${gray},${gray},${gray})`;l.style.strokeWidth=0.6+0.9*rnd();});
    s.querySelectorAll('text.mn').forEach(t=>t.style.display=rnd()<0.5?'none':'');s.querySelectorAll('rect.brk,rect.msel,rect.bhit,rect.chit,circle.hh').forEach(t=>t.style.display='none');   // Klickflächen nie zeigen
    document.querySelectorAll('body *').forEach(el=>{if(getComputedStyle(el).position==='fixed')el.style.display='none';});
    s.querySelectorAll('[style]').forEach(el=>{const st=el.getAttribute('style');if(/var\(--(extra|muted|accent|zug|druck)\)/.test(st))el.setAttribute('style',st.replace(/var\(--(extra|muted|accent|zug|druck)\)/g,'#000'));});
    s.querySelectorAll('.cur,.selx,.forced').forEach(el=>el.classList.remove('cur','selx','forced'));
    const es=0.85+0.3*rnd(),xs=0.8+0.35*rnd(),xsw=1.2+1.4*rnd(),tw=(1.1+0.4*rnd())*d,th=(0.9+0.3*rnd())*d,tsw=0.8+1.0*rnd();
    const items=[];   // [Element, Klasse]
    s.querySelectorAll('g.gn').forEach(g=>{
      [...g.querySelectorAll('ellipse')].forEach(el=>{el.setAttribute('rx',5.4*es);el.setAttribute('ry',3.9*es);const st=el.getAttribute('style')||'';items.push([el,/fill:none/.test(st)?1:0]);});
      [...g.querySelectorAll('path')].forEach(p=>{const m=(p.getAttribute('d')||'').match(/^M([-\d.]+) ([-\d.]+) L([-\d.]+) ([-\d.]+) M/);if(!m)return;
        const cx=(+m[1]+ +m[3])/2,cy=(+m[2]+ +m[4])/2,hollow=+(p.getAttribute('stroke-width')||2)<2;
        const circ=[...g.querySelectorAll('circle:not(.hh)')].find(c=>Math.abs(+c.getAttribute('cx')-cx)<0.6&&Math.abs(+c.getAttribute('cy')-cy)<0.6);
        const wahl=art===1?(rnd()<0.6?'t':'x'):art===2?(hollow?'x':(rnd()<0.75?'p':'x')):art===3?['x','t','p'][Math.floor(rnd()*3)]:'x';
        if(wahl==='t'){   // Dreieck, Spitze oben, hohl ab der Halben
          if(circ)circ.remove();const poly=document.createElementNS(NS,'polygon');
          poly.setAttribute('points',`${cx-tw/2},${cy+th/2} ${cx+tw/2},${cy+th/2} ${cx},${cy-th/2}`);
          poly.setAttribute('style',hollow?`fill:none;stroke:#000;stroke-width:${tsw}`:'fill:#000;stroke:#000;stroke-width:0.5');
          p.replaceWith(poly);items.push([poly,hollow?5:4]);}
        else if(wahl==='p'){   // runder Kopf mit vorgesetztem Doppelkreuz (Rosenzopf)
          if(circ)circ.remove();const e=document.createElementNS(NS,'ellipse');e.setAttribute('cx',cx);e.setAttribute('cy',cy);e.setAttribute('rx',5.4*es);e.setAttribute('ry',3.9*es);
          e.setAttribute('transform',`rotate(-20 ${cx} ${cy})`);e.setAttribute('style','fill:#000');p.replaceWith(e);items.push([e,0]);
          const gx=cx-(1.25+0.35*rnd())*d,r=0.32*d,gp=document.createElementNS(NS,'g');
          const ln=(x1,y1,x2,y2,w)=>{const l=document.createElementNS(NS,'line');l.setAttribute('x1',x1);l.setAttribute('y1',y1);l.setAttribute('x2',x2);l.setAttribute('y2',y2);l.setAttribute('style',`stroke:#000;stroke-width:${w}`);gp.appendChild(l);};
          const w0=0.18*d+0.1*d*rnd();ln(gx-r,cy-r,gx+r,cy+r,w0);ln(gx-r,cy+r,gx+r,cy-r,w0);
          for(const [sx,sy] of [[-1,-1],[1,-1],[-1,1],[1,1]]){const q=document.createElementNS(NS,'rect');const a=(0.24+0.08*rnd())*d;q.setAttribute('x',gx+sx*r-a/2);q.setAttribute('y',cy+sy*r-a/2);q.setAttribute('width',a);q.setAttribute('height',a);q.setAttribute('style','fill:#000');gp.appendChild(q);}
          g.appendChild(gp);items.push([gp,6]);}
        else{const rr=4.5*xs;p.setAttribute('d',`M${cx-rr} ${cy-rr} L${cx+rr} ${cy+rr} M${cx-rr} ${cy+rr} L${cx+rr} ${cy-rr}`);p.setAttribute('stroke-width',hollow?1.3:xsw);
          if(hollow&&!circ){const c=document.createElementNS(NS,'circle');c.setAttribute('cx',cx);c.setAttribute('cy',cy);c.setAttribute('r',0.62*d);c.setAttribute('style','fill:none;stroke:#000;stroke-width:1');g.appendChild(c);}
          items.push([p,hollow?3:2]);}});});
    const R=s.getBoundingClientRect();
    const boxes=items.map(([el,c])=>{const r=el.getBoundingClientRect();return [c,r.left-R.left,r.top-R.top,r.width,r.height];}).filter(q=>q[3]>1&&q[4]>1);
    return {boxes,w:R.width,h:R.height,staff:ys};},[i,Math.floor(rnd()*100000),art]);
   if(!gt||!gt.boxes.length)continue;
   const el=(await pg.$$('#griff svg.gsys'))[i];const fn=`${out}/y${seed0}_${String(k).padStart(5,'0')}`;await el.screenshot({path:fn+'.png'});
   fs.writeFileSync(fn+'.json',JSON.stringify({pc,dsf,art,...gt}));k++;}}
 await pg.close();}
 console.log('Systeme',k);await b.close();})();


In [ ]:
%%writefile /content/yolo_daten.py
# Trainingsdaten für den Kopf-Erkenner (YOLO) aus den Systembildern von yolo_erzeuge_wandlerbilder.js:
# mehrere Systeme werden zu einer Seite gestapelt, die Seite wird zufällig wie ein Scan, eine Kopie oder ein Handyfoto verschlechtert
# (Unschärfe, Rauschen, ungleiche Helligkeit, leichte Drehung und Verzerrung, dünnere oder dickere Striche, JPEG) und in Ausschnitte
# von 960 × 960 Bildpunkten geschnitten. Ausgabe im YOLO-Format (images/, labels/, data.yaml).
# Aufruf: python3 yolo_daten.py ROHORDNER AUSGABEORDNER [SEITEN]
import sys, os, json, glob, random
import numpy as np, cv2

KLASSEN = ['rund_voll', 'rund_hohl', 'kreuz', 'kreuz_kreis', 'dreieck_voll', 'dreieck_hohl', 'vorkreuz']
T = 960   # Ausschnittgröße


def lade(raw):
    out = []
    for f in sorted(glob.glob(os.path.join(raw, '*.json'))):
        j = json.load(open(f)); im = cv2.imread(f[:-5] + '.png', cv2.IMREAD_GRAYSCALE)
        if im is None or not j.get('boxes'): continue
        s = im.shape[1] / j['w']
        out.append((im, [(c, x * s, y * s, w * s, h * s) for c, x, y, w, h in j['boxes']]))
    return out


def seite(sys_, rnd):
    """4–8 Systeme untereinander auf weißem Blatt, auf eine gemeinsame Breite gebracht."""
    W = rnd.randint(1500, 2300); teile = []; boxen = []; y = rnd.randint(60, 250)
    for im, bx in rnd.sample(sys_, k=min(len(sys_), rnd.randint(4, 8))):
        k = W / im.shape[1] * rnd.uniform(0.85, 1.0); im2 = cv2.resize(im, None, fx=k, fy=k, interpolation=cv2.INTER_AREA)
        x0 = rnd.randint(20, max(21, W - im2.shape[1] + 20))
        teile.append((im2, x0, y)); boxen += [(c, x0 + x * k, y + yy * k, w * k, h * k) for c, x, yy, w, h in bx]
        y += im2.shape[0] + rnd.randint(10, 120)
    H = y + rnd.randint(60, 250); pg = np.full((H, W + 60), 255, np.uint8)
    for im2, x0, y0 in teile:
        h, w = im2.shape; pg[y0:y0 + h, x0:x0 + w] = np.minimum(pg[y0:y0 + h, x0:x0 + w], im2[:, :min(w, pg.shape[1] - x0)])
    return pg, boxen


def verschlechtern(pg, boxen, rnd):
    art = rnd.choice(['sauber', 'scan', 'kopie', 'foto', 'foto'])
    g = pg.astype(np.float32)
    if art in ('kopie', 'foto'):   # Striche dünner oder dicker
        k = np.ones((2, 2), np.uint8)
        g = cv2.erode(g, k) if rnd.random() < 0.5 else cv2.dilate(g, k)
    if art != 'sauber':
        g = cv2.GaussianBlur(g, (0, 0), rnd.uniform(0.3, 1.6 if art == 'foto' else 0.9))
    H, W = g.shape
    M = np.eye(3, dtype=np.float32)
    if art in ('scan', 'kopie', 'foto'):   # leichte Drehung, bei Fotos auch Verzerrung
        a = rnd.uniform(-2.5, 2.5) if art == 'foto' else rnd.uniform(-1, 1)
        R = cv2.getRotationMatrix2D((W / 2, H / 2), a, 1.0); M = np.vstack([R, [0, 0, 1]]).astype(np.float32)
        if art == 'foto':
            d = 0.03 * min(W, H); src = np.float32([[0, 0], [W, 0], [W, H], [0, H]])
            dst = src + np.float32([[rnd.uniform(-d, d), rnd.uniform(-d, d)] for _ in range(4)])
            M = cv2.getPerspectiveTransform(src, dst) @ M
        g = cv2.warpPerspective(g, M, (W, H), borderValue=255)
    if art == 'foto':   # ungleiche Helligkeit, grauer Hintergrund, Rauschen
        yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
        grad = 0.55 + 0.4 * (np.cos(xx / W * rnd.uniform(1, 4) + rnd.uniform(0, 6)) * 0.5 + 0.5) * (np.cos(yy / H * rnd.uniform(1, 4)) * 0.5 + 0.5)
        g = g * grad + rnd.uniform(-20, 10)
        g = g + np.random.normal(0, rnd.uniform(3, 12), g.shape)
        g = (g - g.min()) / max(1e-3, g.max() - g.min()) * rnd.uniform(160, 255) + rnd.uniform(0, 40)
    elif art in ('scan', 'kopie'):
        g = g + np.random.normal(0, rnd.uniform(1, 6), g.shape)
    g = np.clip(g, 0, 255).astype(np.uint8)
    if art != 'sauber':
        q = rnd.randint(35, 90); g = cv2.imdecode(cv2.imencode('.jpg', g, [cv2.IMWRITE_JPEG_QUALITY, q])[1], cv2.IMREAD_GRAYSCALE)
    nb = []
    for c, x, y, w, h in boxen:   # Rahmen mitbewegen (Ecken umrechnen, umschließendes Rechteck)
        pts = np.float32([[x, y], [x + w, y], [x + w, y + h], [x, y + h]]).reshape(-1, 1, 2)
        p2 = cv2.perspectiveTransform(pts, M).reshape(-1, 2)
        x0, y0 = p2.min(0); x1, y1 = p2.max(0); nb.append((c, x0, y0, x1 - x0, y1 - y0))
    return g, nb


def ausschnitte(g, boxen, rnd, n):
    H, W = g.shape; out = []
    for _ in range(n):
        x0 = rnd.randint(0, max(0, W - T)); y0 = rnd.randint(0, max(0, H - T))
        crop = np.full((T, T), 255, np.uint8); c_ = g[y0:y0 + T, x0:x0 + T]; crop[:c_.shape[0], :c_.shape[1]] = c_
        lab = []
        for c, x, y, w, h in boxen:
            cx, cy = x + w / 2 - x0, y + h / 2 - y0
            if 0 <= cx < T and 0 <= cy < T and x - x0 > -0.3 * w and x + w - x0 < T + 0.3 * w:
                lab.append(f"{c} {cx / T:.6f} {cy / T:.6f} {min(w, T) / T:.6f} {min(h, T) / T:.6f}")
        if lab: out.append((crop, lab))
    return out


if __name__ == '__main__':
    raw, ziel = sys.argv[1], sys.argv[2]; n_seiten = int(sys.argv[3]) if len(sys.argv) > 3 else 600
    rnd = random.Random(7); np.random.seed(7)
    sys_ = lade(raw); print('Systeme', len(sys_))
    for teil in ('train', 'val'):
        os.makedirs(os.path.join(ziel, 'images', teil), exist_ok=True); os.makedirs(os.path.join(ziel, 'labels', teil), exist_ok=True)
    k = 0
    for i in range(n_seiten):
        pg, bx = seite(sys_, rnd); g, bx = verschlechtern(pg, bx, rnd)
        teil = 'val' if i % 10 == 0 else 'train'
        for crop, lab in ausschnitte(g, bx, rnd, 3):
            fn = f"s{i:05d}_{k:06d}"; cv2.imwrite(os.path.join(ziel, 'images', teil, fn + '.jpg'), crop, [cv2.IMWRITE_JPEG_QUALITY, 92])
            open(os.path.join(ziel, 'labels', teil, fn + '.txt'), 'w').write('\n'.join(lab) + '\n'); k += 1
    with open(os.path.join(ziel, 'data.yaml'), 'w') as f:
        f.write(f"path: {os.path.abspath(ziel)}\ntrain: images/train\nval: images/val\nnames:\n" + ''.join(f"  {i}: {n}\n" for i, n in enumerate(KLASSEN)))
    print('Ausschnitte', k)


In [ ]:
#@title 1. Programme einrichten (ca. 5 Minuten)
!git clone -q --depth 1 https://github.com/jpascher/Harmonikabau.git
%cd /content
!pip -q install ultralytics opencv-python-headless
!npm -s i playwright@1 > /dev/null
!npx -y playwright install --with-deps chromium > /dev/null
import subprocess, torch
print('Grafikkarte:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KEINE – bitte T4-GPU einstellen')

In [ ]:
#@title 2. Trainingsbilder mit dem Wandler setzen (ca. 30–40 Minuten)
ANZAHL_STUECKE = 60  #@param {type:"integer"}
import subprocess, time, os
srv = subprocess.Popen(['python3','-m','http.server','8798'], cwd='/content/Harmonikabau/griffschrift-wandler', stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
GEN='/content/yolo_erzeuge_wandlerbilder.js'
os.makedirs('/content/raw', exist_ok=True)
# zwei Läufe mit verschiedenen Startwerten, damit andere Stücke und Schreibweisen drankommen
for seed in (11, 23):
    !cd /content && NODE_PATH=/content/node_modules node {GEN} /content/raw {seed} {ANZAHL_STUECKE//2}
!ls /content/raw | wc -l

In [ ]:
#@title 3. Seiten zusammenstellen und verschlechtern (ca. 10 Minuten)
SEITEN = 1500  #@param {type:"integer"}
!python3 /content/yolo_daten.py /content/raw /content/ds {SEITEN}

In [ ]:
#@title 4. Lernen (ca. 60–90 Minuten auf einer T4)
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data='/content/ds/data.yaml', imgsz=960, epochs=60, batch=16, patience=15, project='/content/lauf', name='koepfe', exist_ok=True,
            degrees=0, fliplr=0, flipud=0, mosaic=0.5, hsv_h=0, hsv_s=0, hsv_v=0.3, scale=0.25, translate=0.05)
metrics = model.val()
print(metrics.box.map50, metrics.box.map)

In [ ]:
#@title 5. Als ONNX speichern und herunterladen
best = YOLO('/content/lauf/koepfe/weights/best.pt')
best.export(format='onnx', imgsz=960, opset=12, simplify=True)
!cd /content/lauf/koepfe && cp weights/best.pt griffschrift_koepfe.pt && cp weights/best.onnx griffschrift_koepfe.onnx && zip -q -r /content/griffschrift_koepfe.zip griffschrift_koepfe.pt griffschrift_koepfe.onnx results.csv results.png confusion_matrix.png args.yaml
from google.colab import files
files.download('/content/griffschrift_koepfe.zip')